Importamos as bibliotecas usadas para ler os arquivos, organizar as tabelas e realizar os cálculos numéricos.

In [30]:
# Bibliotecas usadas na leitura dos dados e nos calculos.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display


Lemos as datas e os critérios do experimento no arquivo de parâmetros. A formação usa 2017 a 2019; a avaliação de 2020 a 2022 será realizada por outro integrante. O limite de presença em negociação fica definido neste notebook como 90% e vale para os dois estudos.

In [31]:
# Le as datas e os criterios definidos para o experimento.
arquivo_parametros = Path('parametros_experimento.json')
with arquivo_parametros.open(encoding='utf-8') as arquivo:
    parametros = json.load(arquivo)

inicio_formacao = parametros['inicio_formacao']
fim_formacao = parametros['fim_formacao']
inicio_avaliacao = parametros['inicio_avaliacao']
fim_avaliacao = parametros['fim_avaliacao']

# Exige negociacao em pelo menos 90% das datas da formacao nos dois estudos.
presenca_minima_formacao = 0.90

# Mostra os periodos sem misturar mensagens sobre arquivos e downloads.
print('Formacao:', inicio_formacao, 'a', fim_formacao)
print('Avaliacao:', inicio_avaliacao, 'a', fim_avaliacao)


Formacao: 2017-01-01 a 2019-12-31
Avaliacao: 2020-01-01 a 2022-12-31


Executamos o notebook de preparação financeira para disponibilizar os preços, a lista de ações e as funções auxiliares. Os preços locais são reutilizados quando já existem; sem essa cópia, o notebook consulta a fonte.

In [32]:
# Carrega os precos e as funcoes do notebook de preparacao financeira.
%run ./Tabelas_Acoes.ipynb


Formacao da MST: 2017-01-01 ate 2019-12-31
Avaliacao financeira (outro integrante): 2020-01-01 ate 2022-12-31
Pasta da coleta: C:\Users\cida3\OneDrive\Documentos\TrabalhoDiscreta\dados\pandemia
Coleta local ainda nao esta completa. Esta celula nao realiza downloads.


$CASN3.SA: possibly delisted; no price data found  (1d 2008-01-01 -> 2026-01-01)
$AZUL3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$AUAU3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400

3 Failed downloads:
['CASN3.SA']: possibly delisted; no price data found  (1d 2008-01-01 -> 2026-01-01)
['AZUL3.SA', 'AUAU3.SA']: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$EGGY3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$CYRE4.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$HETA3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400

3 Failed downloads:
['EGGY3.SA', 'CYRE4.SA', 'HETA3.SA']: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$ODER4.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$RAIZ4.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$PASS3.SA: Data doesn't exist for startD

Nova coleta salva: dados\pandemia\acoes.csv
Data da coleta (UTC): 2026-11-19T00:46:25.144938+00:00


c:\Users\cida3\anaconda3\envs\AED082026\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
c:\Users\cida3\anaconda3\envs\AED082026\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


Preparamos os retornos logarítmicos e a tabela de elegibilidade usando apenas informações disponíveis até o fim da formação. Além da cobertura de preços e dos critérios anteriores, exigimos volume positivo e finito em pelo menos 90% das datas de referência. A coluna Presenca_negociacao distingue negociação de preço preenchido; valores abaixo do limite geram Excluida com o motivo registrado. As datas da base continuam sendo uma referência provisória de sessões, não um calendário oficial.

In [33]:
def calcular_presenca_negociacao(volumes):
    # Volume positivo indica negociacao; datas sem volume continuam no denominador.
    negociacoes = (volumes > 0) & np.isfinite(volumes)
    return negociacoes.mean()


def preparar_formacao(precos, volumes, inicio, fim, cobertura_minima=0.9,
                      presenca_minima=0.9, tickers_solicitados=None, sessoes=None):
    # Limita os precos ao passado e preserva a tabela original.
    inicio, fim = pd.Timestamp(inicio), pd.Timestamp(fim)
    precos = precos.sort_index().loc[:fim].copy()
    tickers = set(precos.columns)
    if tickers_solicitados is not None:
        tickers.update(tickers_solicitados)
    precos = precos.reindex(columns=sorted(tickers))

    # Usa as datas da base, ou o calendario recebido, como referencia.
    referencia = 'datas da base (provisorio)' if sessoes is None else 'calendario fornecido'
    if sessoes is None:
        sessoes = precos.index
    sessoes = pd.DatetimeIndex(sessoes).unique().sort_values()
    sessoes = sessoes[(sessoes >= inicio) & (sessoes <= fim)]
    precos = precos.reindex(precos.index.union(sessoes).sort_values())

    # Calcula retornos somente para fechamentos positivos e finitos.
    invalidos = precos.notna() & (~np.isfinite(precos) | (precos <= 0))
    precos_validos = precos.where(np.isfinite(precos) & (precos > 0))
    precos_janela = precos_validos.reindex(sessoes)
    retornos = np.log(precos_validos).diff().reindex(sessoes)

    # Mede negociacao somente nas datas e acoes da formacao.
    volumes_formacao = volumes.reindex(index=sessoes, columns=precos.columns)
    presenca_negociacao = calcular_presenca_negociacao(volumes_formacao)

    # Resume a disponibilidade de dados de cada acao.
    quantidade_retornos = retornos.count()
    primeiras_datas = precos_janela.apply(lambda serie: serie.first_valid_index())
    ultimas_datas = precos_janela.apply(lambda serie: serie.last_valid_index())
    tabela = pd.DataFrame({
        'Primeiro_preco_na_formacao': primeiras_datas,
        'Ultimo_preco_na_formacao': ultimas_datas,
        'Historico_no_inicio': precos_validos.loc[:sessoes[0]].notna().any(),
        'Preco_na_ultima_sessao': precos_janela.iloc[-1].notna(),
        'Retornos_validos': quantidade_retornos,
        'Sessoes_referencia': len(sessoes),
        'Cobertura': quantidade_retornos / len(sessoes),
        'Presenca_negociacao': presenca_negociacao,
        'Variancia': retornos.var(),
        'Precos_invalidos': invalidos.reindex(sessoes).sum()
    })
    tabela.index.name = 'Ticker'
    tabela['Situacao'] = 'Elegivel'
    tabela['Motivo'] = 'Atende aos criterios observaveis na base.'

    # Aplica as regras da pesquisa e registra o motivo de cada decisao.
    for ticker, linha in tabela.iterrows():
        exclusoes, pendencias = [], []
        if linha['Retornos_validos'] < 2:
            exclusoes.append('Menos de dois retornos validos')
        elif not np.isfinite(linha['Variancia']) or linha['Variancia'] <= 0:
            exclusoes.append('Variancia nula ou indefinida')
        if linha['Cobertura'] < cobertura_minima:
            exclusoes.append('Cobertura abaixo do minimo')
        if linha['Presenca_negociacao'] < presenca_minima:
            exclusoes.append(f'Presenca de negociacao abaixo de {presenca_minima:.0%}')
        if not linha['Historico_no_inicio']:
            pendencias.append('Sem evidencia de negociacao no inicio')
        if not linha['Preco_na_ultima_sessao']:
            pendencias.append('Sem preco na ultima sessao')
        if linha['Precos_invalidos'] > 0:
            pendencias.append('Precos nao positivos ou infinitos')
        if exclusoes or pendencias:
            tabela.loc[ticker, 'Situacao'] = 'Excluida' if exclusoes else 'Pendente'
            tabela.loc[ticker, 'Motivo'] = '; '.join(exclusoes + pendencias)

    tabela.attrs['referencia_calendario'] = referencia
    return tabela, retornos


Aplicamos os critérios aos preços e volumes de 2017 a 2019 e separamos os retornos das ações elegíveis. A tabela mostra cobertura, presença de negociação e os motivos de inclusão, exclusão ou pendência; a presença é uma proporção de zero a um, de modo que 0,90 corresponde a 90%. Todas as matrizes e carteiras seguintes partem dessa mesma lista filtrada.

In [34]:
# Prepara os retornos e a tabela de elegibilidade com os dados carregados.
tabela_elegibilidade, retornos_formacao = preparar_formacao(
    df['Close'],
    df['Volume'],
    inicio_formacao,
    fim_formacao,
    cobertura_minima=parametros['cobertura_minima_formacao'],
    presenca_minima=presenca_minima_formacao,
    tickers_solicitados=acoes
)

# Mantem somente as acoes que atendem aos criterios da formacao.
acoes_elegiveis = tabela_elegibilidade.index[
    tabela_elegibilidade['Situacao'] == 'Elegivel'
].tolist()
retornos_formacao_elegiveis = retornos_formacao[acoes_elegiveis].copy()

print(tabela_elegibilidade['Situacao'].value_counts())
display(tabela_elegibilidade)


Situacao
Excluida    218
Elegivel    157
Pendente      2
Name: count, dtype: int64


,Primeiro_preco_na_formacao,Ultimo_preco_na_formacao,Historico_no_inicio,Preco_na_ultima_sessao,Retornos_validos,Sessoes_referencia,Cobertura,Presenca_negociacao,Variancia,Precos_invalidos,Situacao,Motivo
Ticker,,,,,,,,,,,,
AALR3.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.987968,0.000274,0,Elegivel,Atende aos criterios observaveis na base.
ABCB4.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.986631,0.000294,0,Elegivel,Atende aos criterios observaveis na base.
ABEV3.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.986631,0.000194,0,Elegivel,Atende aos criterios observaveis na base.
AERI3.SA,NaT,NaT,False,False,0,748,0.00000,0.000000,NaN,0,Excluida,Menos de dois retornos validos; Cobertura abai...
AFLT3.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.466578,0.004316,0,Excluida,Presenca de negociacao abaixo de 90%
...,...,...,...,...,...,...,...,...,...,...,...,...
WHRL4.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.836898,0.000705,0,Excluida,Presenca de negociacao abaixo de 90%
WIZC3.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.985294,0.000716,0,Elegivel,Atende aos criterios observaveis na base.
WLMM3.SA,2017-07-07,2019-12-30,False,True,619,748,0.82754,0.177807,0.001762,0,Excluida,Cobertura abaixo do minimo; Presenca de negoci...


Calculamos a correlação de Pearson entre os retornos das ações elegíveis, exigindo o mínimo configurado de observações conjuntas por par. O padrão é 80% das datas da formação.

In [35]:
# Exige observacoes conjuntas em pelo menos 80% das datas da formacao.
proporcao_minima = parametros['proporcao_minima_observacoes_conjuntas']
min_observacoes_conjuntas = int(np.ceil(
    len(retornos_formacao_elegiveis) * proporcao_minima
))

# Calcula Pearson entre os retornos das acoes elegiveis.
matriz_correlacoes_formacao = retornos_formacao_elegiveis.corr(
    method='pearson',
    min_periods=min_observacoes_conjuntas
)
matriz_correlacoes_formacao


Ticker,AALR3.SA,ABCB4.SA,ABEV3.SA,AGRO3.SA,ALOS3.SA,ALPA4.SA,AMAR3.SA,AMER3.SA,ANIM3.SA,AXIA3.SA,...,USIM5.SA,VALE3.SA,VIVR3.SA,VIVT3.SA,VLID3.SA,VSTE3.SA,VULC3.SA,WEGE3.SA,WIZC3.SA,YDUQ3.SA
Ticker,,,,,,,,,,,,,,,,,,,,,
AALR3.SA,1.000000,0.202691,0.110106,0.083589,0.121380,0.179266,0.155251,0.183235,0.115109,0.165066,...,0.145422,0.090904,-0.006863,0.013556,0.085702,0.045009,0.111357,0.046897,0.109219,0.126002
ABCB4.SA,0.202691,1.000000,0.189648,0.220784,0.261322,0.327922,0.308507,0.356325,0.289055,0.378370,...,0.370741,0.128522,0.004433,0.168491,0.264149,0.185372,0.219497,0.297668,0.141744,0.366786
ABEV3.SA,0.110106,0.189648,1.000000,0.114369,0.180408,0.201264,0.191764,0.213857,0.168677,0.176474,...,0.292719,0.109521,0.023429,0.203723,0.114467,0.076082,0.141875,0.298128,0.095805,0.256555
AGRO3.SA,0.083589,0.220784,0.114369,1.000000,0.143613,0.137455,0.153953,0.138819,0.174440,0.136560,...,0.210528,0.096113,0.023992,0.171983,0.176458,0.112741,0.147901,0.114665,0.136881,0.221728
ALOS3.SA,0.121380,0.261322,0.180408,0.143613,1.000000,0.230432,0.304599,0.250159,0.161043,0.268904,...,0.271224,0.098270,0.029677,0.148716,0.185005,0.166616,0.164560,0.123376,0.116523,0.174962
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VSTE3.SA,0.045009,0.185372,0.076082,0.112741,0.166616,0.168694,0.201989,0.089922,0.086537,0.170186,...,0.135063,-0.002527,0.023101,0.119548,0.097366,1.000000,0.191360,0.070860,0.127214,0.106222
VULC3.SA,0.111357,0.219497,0.141875,0.147901,0.164560,0.279568,0.208381,0.264659,0.256361,0.211275,...,0.288955,0.155577,0.018543,0.170352,0.224925,0.191360,1.000000,0.202989,0.138996,0.282941
WEGE3.SA,0.046897,0.297668,0.298128,0.114665,0.123376,0.247620,0.227212,0.241121,0.164563,0.208062,...,0.285675,0.177610,0.088669,0.254705,0.235154,0.070860,0.202989,1.000000,0.099615,0.271211


Transformamos as correlações em distâncias de Mantegna. Cada ação representa um vértice, e cada distância conhecida será o peso de uma aresta.

In [36]:
def matriz_mantegna(matriz_pearson):
    # Correlacoes maiores produzem distancias menores.
    return np.sqrt(2 * (1 - matriz_pearson))

matriz_mantegna_formacao = matriz_mantegna(matriz_correlacoes_formacao)
matriz_mantegna_formacao


Ticker,AALR3.SA,ABCB4.SA,ABEV3.SA,AGRO3.SA,ALOS3.SA,ALPA4.SA,AMAR3.SA,AMER3.SA,ANIM3.SA,AXIA3.SA,...,USIM5.SA,VALE3.SA,VIVR3.SA,VIVT3.SA,VLID3.SA,VSTE3.SA,VULC3.SA,WEGE3.SA,WIZC3.SA,YDUQ3.SA
Ticker,,,,,,,,,,,,,,,,,,,,,
AALR3.SA,0.000000,1.262782,1.334087,1.353818,1.325609,1.281198,1.299807,1.278096,1.330331,1.292234,...,1.307347,1.348404,1.419058,1.404595,1.352256,1.382021,1.333149,1.380654,1.334752,1.322118
ABCB4.SA,1.262782,0.000000,1.273068,1.248372,1.215465,1.159378,1.176004,1.134614,1.192430,1.115016,...,1.121837,1.320211,1.411075,1.289581,1.213138,1.276423,1.249403,1.185185,1.310157,1.125357
ABEV3.SA,1.334087,1.273068,0.000000,1.330888,1.280306,1.263911,1.271406,1.253909,1.289436,1.283375,...,1.189354,1.334525,1.397549,1.261964,1.330814,1.359351,1.310058,1.184797,1.344764,1.219381
AGRO3.SA,1.353818,1.248372,1.330888,0.000000,1.308730,1.313427,1.300805,1.312388,1.284959,1.314108,...,1.256561,1.344535,1.397146,1.286870,1.283388,1.332110,1.305450,1.330665,1.313864,1.247615
ALOS3.SA,1.325609,1.215465,1.280306,1.308730,0.000000,1.240619,1.179323,1.224615,1.295343,1.209211,...,1.207291,1.342930,1.393071,1.304825,1.276711,1.291034,1.292625,1.324103,1.329268,1.284553
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VSTE3.SA,1.382021,1.276423,1.359351,1.332110,1.291034,1.289423,1.263338,1.349132,1.351638,1.288266,...,1.315247,1.416000,1.397783,1.326991,1.343602,0.000000,1.271723,1.363188,1.321201,1.336995
VULC3.SA,1.333149,1.249403,1.310058,1.305450,1.292625,1.200360,1.258268,1.212717,1.219540,1.255966,...,1.192514,1.299556,1.401040,1.288137,1.245050,1.271723,0.000000,1.262546,1.312253,1.197547
WEGE3.SA,1.380654,1.185185,1.184797,1.330665,1.324103,1.226687,1.243212,1.231973,1.292623,1.258521,...,1.195261,1.282490,1.350060,1.220898,1.236807,1.363188,1.262546,0.000000,1.341928,1.207302


Mantemos esta análise de IPCA como exploração separada. Ela correlaciona níveis de preços corrigidos e não participa da MST principal; pode ser ignorada ao executar o experimento da formação.

In [37]:
# Mantem a analise exploratoria de IPCA separada da MST principal.
dataframe_IPCA = inflacao_mensal('01/01/2008', '31/12/2025', variacao_logaritma)
matriz_correlacao_IPCA = dataframe_IPCA.corr(method='pearson')
matriz_mantegna_IPCA = matriz_mantegna(matriz_correlacao_IPCA)
matriz_mantegna_IPCA


Ticker,AALR3.SA,ABCB4.SA,ABEV3.SA,AERI3.SA,AFLT3.SA,AGRO3.SA,AGXY3.SA,AHEB3.SA,AHEB5.SA,AHEB6.SA,...,VVEO3.SA,WDCN3.SA,WEGE3.SA,WEST3.SA,WHRL3.SA,WHRL4.SA,WIZC3.SA,WLMM3.SA,WLMM4.SA,YDUQ3.SA
Ticker,,,,,,,,,,,,,,,,,,,,,
AALR3.SA,0.000000,1.704417,0.924978,1.294628,1.454682,1.669608,0.677664,1.196351,1.172692,1.120467,...,0.745838,1.140178,1.757616,1.379627,1.524702,1.541576,1.100217,1.661687,1.722044,1.166335
ABCB4.SA,1.704417,0.000000,1.279995,1.865418,1.411735,0.704319,1.926850,1.362498,1.339858,1.225228,...,1.879404,1.843638,0.692761,1.797131,0.876629,0.889326,1.605027,1.257971,1.012812,1.230195
ABEV3.SA,0.924978,1.279995,0.000000,0.611612,1.851240,1.526964,0.809750,1.224491,1.087871,0.972039,...,0.738861,0.723292,1.434524,0.704791,1.244488,1.271774,0.950919,1.800141,1.489921,0.897991
AERI3.SA,1.294628,1.865418,0.611612,0.000000,0.746054,1.463357,0.712185,1.365848,1.596775,1.132868,...,0.785605,0.223225,1.424814,0.379281,0.345908,0.400217,0.870020,1.076707,0.997271,0.485283
AFLT3.SA,1.454682,1.411735,1.851240,0.746054,0.000000,1.209491,0.670768,1.459405,1.630692,1.583092,...,0.660648,0.988693,1.313996,0.842420,1.329453,1.306235,1.508114,1.129972,1.362239,1.478826
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WHRL4.SA,1.541576,0.889326,1.271774,0.400217,1.306235,0.788983,1.011034,1.440111,1.507873,1.224668,...,1.048162,0.443401,0.708333,0.571198,0.174015,0.000000,1.384091,1.098920,0.850889,0.903323
WIZC3.SA,1.100217,1.605027,0.950919,0.870020,1.508114,1.702367,1.126504,1.283783,1.199908,1.015482,...,1.104330,0.697013,1.690329,0.809730,1.398500,1.384091,0.000000,1.474599,1.639765,1.058242
WLMM3.SA,1.661687,1.257971,1.800141,1.076707,1.129972,0.596167,1.161864,1.808083,1.840540,1.772861,...,1.261292,0.734490,0.656380,1.107470,1.155712,1.098920,1.474599,0.000000,0.377054,1.709209


Criamos uma lista de arestas no formato peso, vértice inicial e vértice final. Cada par aparece uma vez, distâncias ausentes são ignoradas e a lista é ordenada por peso crescente.

In [38]:
def criar_arestas(df):
    arestas = []
    n = len(df.columns)

    # Percorre apenas um triangulo da matriz, sem repetir pares.
    for j in range(1, n):
        for i in range(j):
            peso = df.iloc[i, j]
            # Uma distancia desconhecida nao representa uma aresta de peso zero.
            if np.isfinite(peso):
                arestas.append((peso, i, j))

    # A tupla desempata pesos iguais pelos indices dos vertices.
    return sorted(arestas)

arestas = criar_arestas(matriz_mantegna_formacao)
# Mostra uma amostra para nao imprimir milhares de arestas.
arestas[:10]


[(np.float64(0.32165109904257816), 98, 99),
 (np.float64(0.3669180436190365), 12, 13),
 (np.float64(0.41251448578418404), 72, 74),
 (np.float64(0.45617761874577284), 73, 74),
 (np.float64(0.48565662167849744), 22, 148),
 (np.float64(0.49365061053070863), 30, 31),
 (np.float64(0.49805246651541296), 62, 64),
 (np.float64(0.5183406583199556), 13, 74),
 (np.float64(0.532962506075205), 21, 22),
 (np.float64(0.5569565040388144), 63, 64)]

Construímos a MST com Kruskal e Union-Find. As arestas são consideradas em ordem crescente de peso e aceitas quando ligam componentes diferentes, evitando ciclos; a entrada deve permitir conectar todos os vértices.

In [39]:
def find(lista, vertice):
    # Segue os pais ate encontrar o representante da componente.
    while lista[vertice] != vertice:
        vertice = lista[vertice]
    return vertice


def union(lista, vertice1, vertice2):
    # Liga os representantes de duas componentes diferentes.
    pai1 = find(lista, vertice1)
    pai2 = find(lista, vertice2)
    if pai1 != pai2:
        lista[pai2] = pai1


def kruskal(df):
    arestas = criar_arestas(df)
    n = len(df.columns)
    pais = list(range(n))
    mst = []

    for peso, vertice1, vertice2 in arestas:
        # Aceita a aresta quando ela une componentes, evitando ciclos.
        if find(pais, vertice1) != find(pais, vertice2):
            union(pais, vertice1, vertice2)
            mst.append((peso, vertice1, vertice2))
        # Uma arvore com n vertices possui n - 1 arestas.
        if len(mst) == n - 1:
            break

    return mst


Prim cresce a árvore a partir do vértice zero, escolhendo a menor aresta que alcança um vértice ainda não visitado. A função permanece disponível, mas sua chamada está comentada.

In [40]:
def prim(df):
    n = len(df.columns)
    visitados = {0}
    mst = []

    while len(visitados) < n:
        menor_peso = (np.inf, None, None)
        for u in visitados:
            for v in range(n):
                if v not in visitados:
                    peso = df.iloc[u, v]
                    # Busca a menor ligacao entre visitados e nao visitados.
                    if np.isfinite(peso) and peso < menor_peso[0]:
                        menor_peso = (peso, u, v)

        # Encerra quando nao existe outra ligacao disponivel.
        if menor_peso[2] is None:
            break
        visitados.add(menor_peso[2])
        mst.append(menor_peso)

    return mst


Executamos Kruskal sobre a matriz da formação e mostramos a quantidade de arestas e o peso total. Prim não é executado nesta célula.

In [41]:
# A chamada de Prim permanece desativada.
# mst_prim = prim(matriz_mantegna_formacao)

# Constroi a MST e soma os pesos das arestas escolhidas.
mst_kruskal = kruskal(matriz_mantegna_formacao)
peso_kruskal = sum(peso for peso, u, v in mst_kruskal)

print('Arestas Kruskal:', len(mst_kruskal))
print('Peso total Kruskal:', peso_kruskal)


Arestas Kruskal: 156
Peso total Kruskal: 164.04191180381395


Representamos cada aresta nos dois sentidos e calculamos grau, distância, excentricidade, diâmetro e centro. As distâncias somam os pesos ao longo do caminho único de uma árvore conexa; as funções pressupõem essa entrada.

In [42]:
def criar_dicionario_adjacencia(mst):
    adjacencia = {}
    for peso, v1, v2 in mst:
        # Registra os dois sentidos de cada aresta nao dirigida.
        adjacencia.setdefault(v1, []).append((peso, v2))
        adjacencia.setdefault(v2, []).append((peso, v1))
    return adjacencia


def calcular_graus(adjacencia):
    # O grau de um vertice e a quantidade de vizinhos.
    graus = {}
    for vertice in adjacencia:
        graus[vertice] = len(adjacencia[vertice])
    return graus


def calcular_excentricidade(adjacencia):
    excentricidades = {}
    for origem in adjacencia:
        distancias = {origem: 0}
        pilha = [origem]

        # DFS: soma os pesos ao longo do caminho unico da arvore.
        while pilha:
            vertice = pilha.pop()
            for peso, vizinho in adjacencia[vertice]:
                if vizinho not in distancias:
                    distancias[vizinho] = distancias[vertice] + peso
                    pilha.append(vizinho)

        excentricidades[origem] = max(distancias.values())
    return excentricidades


def distancia_entre_vertices(adjacencia, origem, destino):
    distancias = {origem: 0}
    pilha = [origem]
    while pilha:
        vertice = pilha.pop()
        if vertice == destino:
            return distancias[vertice]
        for peso, vizinho in adjacencia[vertice]:
            if vizinho not in distancias:
                distancias[vizinho] = distancias[vertice] + peso
                pilha.append(vizinho)


def calcular_diametro(adjacencia):
    # O diametro e a maior excentricidade da arvore.
    excentricidades = calcular_excentricidade(adjacencia)
    return max(excentricidades.values())


def calcular_centro(adjacencia):
    excentricidades = calcular_excentricidade(adjacencia)
    menor_excentricidade = min(excentricidades.values())
    centro = []
    for vertice, excentricidade in excentricidades.items():
        # A tolerancia considera pequenos arredondamentos nas somas.
        if abs(excentricidade - menor_excentricidade) <= 1e-10:
            centro.append(vertice)
    return centro


Reunimos as métricas dos vértices e associamos seus índices aos tickers da mesma matriz usada em Kruskal. A tabela mostra grau, excentricidade e identificação de folhas e centro.

In [43]:
# Calcula as metricas usando a arvore produzida por Kruskal.
adjacencia = criar_dicionario_adjacencia(mst_kruskal)
graus = calcular_graus(adjacencia)
excentricidades = calcular_excentricidade(adjacencia)
diametro = calcular_diametro(adjacencia)
centro = calcular_centro(adjacencia)
vertices = sorted(adjacencia)

# Recupera os tickers na mesma ordem da matriz usada no grafo.
metricas = pd.DataFrame({
    'Vertice': vertices,
    'Ticker': [matriz_mantegna_formacao.columns[v] for v in vertices],
    'Grau': [graus[v] for v in vertices],
    'Excentricidade': [excentricidades[v] for v in vertices],
    'Folha': [graus[v] == 1 for v in vertices],
    'Centro': [v in centro for v in vertices]
})

display(metricas.sort_values(['Excentricidade', 'Ticker']))
print('Diametro:', diametro)
print('Centro:', [matriz_mantegna_formacao.columns[v] for v in centro])


,Vertice,Ticker,Grau,Excentricidade,Folha,Centro
83,83,LREN3.SA,6,6.707738,False,True
72,72,ITSA4.SA,8,7.061241,False,False
74,74,ITUB4.SA,8,7.473756,False,False
93,93,MULT3.SA,10,7.588096,False,False
115,115,RENT3.SA,2,7.626816,False,False
...,...,...,...,...,...,...
148,148,VALE3.SA,1,12.315081,True,False
21,21,BRAP3.SA,1,12.362387,True,False
149,149,VIVR3.SA,1,12.855160,True,False
79,79,LEVE3.SA,1,12.873829,True,False


Diametro: 12.87382913852293
Centro: ['LREN3.SA']


Selecionamos as oito ações com menores excentricidades para a carteira central. Empates são resolvidos pelo ticker em ordem alfabética, preservando as demais métricas.

In [44]:
# Ordena da menor para a maior excentricidade, com desempate por ticker.
carteira_central = metricas.sort_values(
    ['Excentricidade', 'Ticker'], ascending=[True, True]
).head(8).copy()

display(carteira_central)


,Vertice,Ticker,Grau,Excentricidade,Folha,Centro
83,83,LREN3.SA,6,6.707738,False,True
72,72,ITSA4.SA,8,7.061241,False,False
74,74,ITUB4.SA,8,7.473756,False,False
93,93,MULT3.SA,10,7.588096,False,False
115,115,RENT3.SA,2,7.626816,False,False
71,71,ITSA3.SA,1,7.735642,True,False
10,10,AZZA3.SA,1,7.737691,True,False
7,7,AMER3.SA,2,7.749695,False,False


Selecionamos as oito ações com maiores excentricidades para a carteira periférica. Empates são resolvidos pelo ticker em ordem alfabética; as duas seleções são independentes e não removem eventual sobreposição.

In [45]:
# Ordena da maior para a menor excentricidade, com desempate por ticker.
carteira_periferica = metricas.sort_values(
    ['Excentricidade', 'Ticker'], ascending=[False, True]
).head(8).copy()

display(carteira_periferica)


,Vertice,Ticker,Grau,Excentricidade,Folha,Centro
79,79,LEVE3.SA,1,12.873829,True,False
95,95,NEXP3.SA,1,12.873829,True,False
149,149,VIVR3.SA,1,12.855160,True,False
21,21,BRAP3.SA,1,12.362387,True,False
148,148,VALE3.SA,1,12.315081,True,False
76,76,KEPL3.SA,1,11.844023,True,False
22,22,BRAP4.SA,3,11.829424,False,False
51,51,ETER3.SA,1,11.827321,True,False


A carteira de controle usa os ativos presentes na MST que pertenciam à carteira recomendada pelo BTG Pactual em abril de 2019, descrita no relatório como mais defensiva e diversificada. A escolha vem da referência externa, sem grau, excentricidade, distância ou desempenho de 2020 a 2022; o filtro da MST considera somente os tickers. Os segmentos seguem o relatório, não uma classificação setorial oficial da B3. A interseção adapta a carteira original e não constitui comprovação de proteção em crises. Fonte: https://content.btgpactual.com/wp-content/uploads/2019/04/10_sim_abr_2019-vfinal.pdf

In [46]:
# Referencia externa publicada antes da avaliacao da pandemia.
# Segmentos informados no relatorio do BTG Pactual de abril de 2019.
fonte_controle = 'https://content.btgpactual.com/wp-content/uploads/2019/04/10_sim_abr_2019-vfinal.pdf'
referencia_controle = pd.DataFrame([
    ('PETR4.SA', 'Oleo e Gas'),
    ('BBDC4.SA', 'Bancos'),
    ('CSAN3.SA', 'Acucar e Etanol'),
    ('IRBR3.SA', 'Financeiro'),
    ('LREN3.SA', 'Varejo'),
    ('RAIL3.SA', 'Infraestrutura'),
    ('RENT3.SA', 'Aluguel de carros'),
    ('SUZB3.SA', 'Papel e Celulose'),
    ('ABEV3.SA', 'Alimentos e Bebidas'),
    ('OIBR3.SA', 'Telecom')
], columns=['Ticker', 'Segmento'])

# Consulta apenas a presenca dos ativos na arvore, sem usar suas metricas.
tickers_mst = {
    matriz_mantegna_formacao.columns[vertice]
    for _, u, v in mst_kruskal
    for vertice in (u, v)
}
presentes = referencia_controle['Ticker'].isin(tickers_mst)
carteira_controle = referencia_controle.loc[presentes].copy()
carteira_controle = carteira_controle.sort_values('Ticker').reset_index(drop=True)

# Mostra a adaptacao da referencia, sem substituir os ativos ausentes.
print('Referencia: BTG Pactual, abril de 2019')
print('Fonte:', fonte_controle)
print('Papeis da referencia fora da MST:', referencia_controle.loc[~presentes, 'Ticker'].tolist())
display(carteira_controle)


Referencia: BTG Pactual, abril de 2019
Fonte: https://content.btgpactual.com/wp-content/uploads/2019/04/10_sim_abr_2019-vfinal.pdf
Papeis da referencia fora da MST: ['IRBR3.SA', 'SUZB3.SA', 'OIBR3.SA']


,Ticker,Segmento
0,ABEV3.SA,Alimentos e Bebidas
1,BBDC4.SA,Bancos
2,CSAN3.SA,Acucar e Etanol
3,LREN3.SA,Varejo
4,PETR4.SA,Oleo e Gas
5,RAIL3.SA,Infraestrutura
6,RENT3.SA,Aluguel de carros


O estudo da crise de 2008 usa janeiro de 2004 a dezembro de 2006 para formar a MST e janeiro de 2007 a dezembro de 2009 como referência da avaliação financeira. As variáveis recebem o sufixo 2008 para preservar os resultados da pandemia.

In [47]:
# Le apenas a configuracao do segundo experimento.
parametros_2008 = parametros['crise_2008'].copy()
inicio_formacao_2008 = parametros_2008['inicio_formacao']
fim_formacao_2008 = parametros_2008['fim_formacao']
inicio_avaliacao_2008 = parametros_2008['inicio_avaliacao']
fim_avaliacao_2008 = parametros_2008['fim_avaliacao']

print('Formacao de 2008:', inicio_formacao_2008, 'a', fim_formacao_2008)
print('Avaliacao de 2008:', inicio_avaliacao_2008, 'a', fim_avaliacao_2008)


Formacao de 2008: 2004-01-01 a 2006-12-31
Avaliacao de 2008: 2007-01-01 a 2009-12-31


Carregamos a coleta de 2008 em uma pasta própria. O histórico começa em dezembro de 2003 para permitir o primeiro retorno de 2004 e termina em dezembro de 2009; os arquivos locais são reutilizados quando já existem. A lista de tickers é a mesma da fonte da pandemia e não constitui um universo histórico completo da bolsa.

In [48]:
# Reutiliza a funcao de carga definida em Tabelas_Acoes.ipynb.
# A coleta e separada; df e as variaveis da pandemia nao sao substituidos.
df_2008, acoes_2008, metadados_coleta_2008 = obter_coleta(
    parametros_2008, acoes_solicitadas=acoes
)
df_2008['Close']


$ANIM3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$BMGB4.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$ALPK3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$CASH3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$AURE3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$CEAB3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$CAML3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$AGXY3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$ASAI3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$CASN3.SA: possibly delisted; no price data found  (1d 2003-12-01 -> 2010-01-01)
$BRST3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$AZTE3.SA: Data doesn't exist for startDate = 1070244000, endDate = 1262311200
$AERI3.SA: Data doesn't exist for startDate = 1070

Nova coleta salva: dados\crise_2008\acoes.csv
Data da coleta (UTC): 2026-11-19T00:47:47.877110+00:00


Ticker,ABCB4.SA,ABEV3.SA,AFLT3.SA,AGRO3.SA,AHEB3.SA,AHEB5.SA,AHEB6.SA,ALPA3.SA,ALPA4.SA,AMAR3.SA,...,VIVR3.SA,VIVT3.SA,VLID3.SA,VSTE3.SA,VULC3.SA,WEGE3.SA,WHRL3.SA,WHRL4.SA,WLMM4.SA,YDUQ3.SA
Date,,,,,,,,,,,,,,,,,,,,,
2003-12-01,NaN,0.945028,NaN,NaN,4.445885,NaN,NaN,0.329332,NaN,NaN,...,NaN,3.832608,NaN,NaN,2453.502441,0.379055,0.144752,0.220319,747.640137,NaN
2003-12-02,NaN,0.942004,NaN,NaN,4.445885,NaN,NaN,0.329332,NaN,NaN,...,NaN,3.748656,NaN,NaN,2453.502441,0.379055,0.144752,0.220319,747.640137,NaN
2003-12-03,NaN,0.922346,NaN,NaN,4.445885,NaN,NaN,0.329332,NaN,NaN,...,NaN,3.821654,NaN,NaN,2453.502441,0.379055,0.144752,0.199447,757.976135,NaN
2003-12-04,NaN,0.922346,NaN,NaN,4.445885,NaN,NaN,0.329332,NaN,NaN,...,NaN,3.808274,NaN,NaN,2453.502441,0.379055,0.144752,0.199447,695.960144,NaN
2003-12-05,NaN,0.922346,NaN,NaN,4.445885,NaN,NaN,0.329332,NaN,NaN,...,NaN,3.832608,NaN,NaN,2453.502441,0.379055,0.144752,0.199447,695.960144,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2009-12-23,3.204377,2.931896,5.293936,4.480400,22.192442,9.84313,10.0,2.533335,2.014050,40.804920,...,217401.171875,4.893388,7.262399,99.939682,15.171151,1.882181,0.892625,0.776914,3.789881,4.854095
2009-12-24,3.204377,2.931896,5.293936,4.480400,22.192442,9.84313,10.0,2.533335,2.014050,40.804920,...,217401.171875,4.893388,7.262399,99.939682,15.171151,1.882181,0.892625,0.776914,3.789881,4.854095
2009-12-28,3.164156,2.947849,5.293936,4.524110,22.192442,9.84313,10.0,2.533335,2.031541,40.461067,...,218027.687500,4.928912,7.277933,100.245781,15.171151,1.893621,0.867830,0.788510,3.789881,4.908636


Aplicamos aos preços e volumes de 2004 a 2006 as mesmas regras de cobertura, qualidade e presença mínima de negociação de 90%. A tabela de 2008 fica separada da tabela da pandemia, e os dados de 2007 a 2009 não participam da elegibilidade. As carteiras central, periférica e de controle são reconstruídas a partir da nova MST, sem completar vagas com ativos excluídos.

In [49]:
# Reaproveita a preparacao da formacao, sem usar precos da avaliacao.
tabela_elegibilidade_2008, retornos_formacao_2008 = preparar_formacao(
    df_2008['Close'],
    df_2008['Volume'],
    inicio_formacao_2008,
    fim_formacao_2008,
    cobertura_minima=parametros_2008['cobertura_minima_formacao'],
    presenca_minima=presenca_minima_formacao,
    tickers_solicitados=acoes_2008
)
acoes_elegiveis_2008 = tabela_elegibilidade_2008.index[
    tabela_elegibilidade_2008['Situacao'] == 'Elegivel'
].tolist()
retornos_formacao_elegiveis_2008 = retornos_formacao_2008[acoes_elegiveis_2008].copy()

print(tabela_elegibilidade_2008['Situacao'].value_counts())
display(tabela_elegibilidade_2008)


Situacao
Excluida    340
Elegivel     37
Name: count, dtype: int64


,Primeiro_preco_na_formacao,Ultimo_preco_na_formacao,Historico_no_inicio,Preco_na_ultima_sessao,Retornos_validos,Sessoes_referencia,Cobertura,Presenca_negociacao,Variancia,Precos_invalidos,Situacao,Motivo
Ticker,,,,,,,,,,,,
AALR3.SA,NaT,NaT,False,False,0,768,0.000000,0.000000,NaN,0,Excluida,Menos de dois retornos validos; Cobertura abai...
ABCB4.SA,NaT,NaT,False,False,0,768,0.000000,0.000000,NaN,0,Excluida,Menos de dois retornos validos; Cobertura abai...
ABEV3.SA,2004-01-01,2006-12-28,True,True,768,768,1.000000,0.945312,0.000402,0,Elegivel,Atende aos criterios observaveis na base.
AERI3.SA,NaT,NaT,False,False,0,768,0.000000,0.000000,NaN,0,Excluida,Menos de dois retornos validos; Cobertura abai...
AFLT3.SA,2006-07-31,2006-12-28,False,True,107,768,0.139323,0.031250,0.005388,0,Excluida,Cobertura abaixo do minimo; Presenca de negoci...
...,...,...,...,...,...,...,...,...,...,...,...,...
WHRL4.SA,2004-01-01,2006-12-28,True,True,768,768,1.000000,0.470052,0.001141,0,Excluida,Presenca de negociacao abaixo de 90%
WIZC3.SA,NaT,NaT,False,False,0,768,0.000000,0.000000,NaN,0,Excluida,Menos de dois retornos validos; Cobertura abai...
WLMM3.SA,NaT,NaT,False,False,0,768,0.000000,0.000000,NaN,0,Excluida,Menos de dois retornos validos; Cobertura abai...


Calculamos Pearson e Mantegna somente com os retornos elegíveis de 2004 a 2006. O mínimo de observações conjuntas continua sendo 80% das datas da formação, conforme o arquivo de parâmetros.

In [50]:
# O minimo e calculado com o numero de datas deste experimento.
proporcao_minima_2008 = parametros_2008['proporcao_minima_observacoes_conjuntas']
min_observacoes_conjuntas_2008 = int(np.ceil(
    len(retornos_formacao_elegiveis_2008) * proporcao_minima_2008
))
matriz_correlacoes_formacao_2008 = retornos_formacao_elegiveis_2008.corr(
    method='pearson', min_periods=min_observacoes_conjuntas_2008
)
matriz_mantegna_formacao_2008 = matriz_mantegna(matriz_correlacoes_formacao_2008)
matriz_mantegna_formacao_2008


Ticker,ABEV3.SA,AXIA3.SA,BBAS3.SA,BBDC3.SA,BRAP3.SA,BRAP4.SA,BRKM5.SA,CGAS5.SA,CMIG3.SA,CMIG4.SA,...,PETR3.SA,PETR4.SA,POMO4.SA,RAPT4.SA,SBSP3.SA,UNIP6.SA,USIM3.SA,USIM5.SA,VALE3.SA,VIVT3.SA
Ticker,,,,,,,,,,,,,,,,,,,,,
ABEV3.SA,0.000000,1.269716,1.292767,1.264761,1.352256,1.274944,1.304090,1.280353,1.324227,1.424265,...,1.253085,1.268835,1.359355,1.280966,1.318831,1.291194,1.300683,1.265118,1.227817,1.321664
AXIA3.SA,1.269716,0.000000,1.055582,1.092594,1.274400,1.129453,1.040870,1.072638,0.987709,1.385660,...,1.101687,1.086217,1.236206,1.209802,1.138899,1.137001,1.180312,0.959055,1.148027,1.155944
BBAS3.SA,1.292767,1.055582,0.000000,1.084874,1.296499,1.162764,1.070271,1.077338,1.128749,1.399775,...,1.161951,1.149919,1.242682,1.209136,1.150305,1.161568,1.229663,0.992403,1.170332,1.167874
BBDC3.SA,1.264761,1.092594,1.084874,0.000000,1.294632,1.154279,1.124197,1.172727,1.105638,1.375404,...,1.128839,1.116174,1.292784,1.247862,1.161944,1.201865,1.212376,1.052983,1.125498,1.186888
BRAP3.SA,1.352256,1.274400,1.296499,1.294632,0.000000,1.161818,1.270553,1.311767,1.248177,1.404020,...,1.253344,1.262445,1.346255,1.311968,1.321991,1.296064,1.263065,1.224756,1.225582,1.265968
BRAP4.SA,1.274944,1.129453,1.162764,1.154279,1.161818,0.000000,1.145693,1.177200,1.085898,1.387117,...,1.068901,1.001253,1.210335,1.188415,1.113344,1.148255,1.188703,1.062136,0.901032,1.184029
BRKM5.SA,1.304090,1.040870,1.070271,1.124197,1.270553,1.145693,0.000000,1.060076,1.100196,1.401207,...,1.186644,1.178065,1.253441,1.187779,1.149250,0.977154,1.144649,0.955476,1.165260,1.154896
CGAS5.SA,1.280353,1.072638,1.077338,1.172727,1.311767,1.177200,1.060076,0.000000,1.062138,1.387784,...,1.201580,1.158113,1.274977,1.266971,1.157332,1.112631,1.217142,1.037785,1.193124,1.141276
CMIG3.SA,1.324227,0.987709,1.128749,1.105638,1.248177,1.085898,1.100196,1.062138,0.000000,1.368622,...,1.145999,1.055912,1.207125,1.216534,1.052340,1.133689,1.178277,1.064361,1.152787,1.180634


Construímos a MST de 2008 com a mesma função de Kruskal, usando a nova matriz. A chamada de Prim continua desativada, e a entrada deve permitir uma árvore conexa.

In [51]:
# Prim permanece disponivel, mas nao e executado.
# mst_prim_2008 = prim(matriz_mantegna_formacao_2008)
mst_kruskal_2008 = kruskal(matriz_mantegna_formacao_2008)
peso_kruskal_2008 = sum(peso for peso, u, v in mst_kruskal_2008)

print('Arestas Kruskal de 2008:', len(mst_kruskal_2008))
print('Peso total Kruskal de 2008:', peso_kruskal_2008)


Arestas Kruskal de 2008: 36
Peso total Kruskal de 2008: 36.072253513524444


Calculamos os graus, as excentricidades, o diâmetro e o centro da árvore de 2008. Os tickers são recuperados da matriz deste experimento, preservando a correspondência entre os nomes das ações e os índices dos vértices.

In [52]:
# Reaproveita as funcoes estruturais sem alterar as da pandemia.
adjacencia_2008 = criar_dicionario_adjacencia(mst_kruskal_2008)
graus_2008 = calcular_graus(adjacencia_2008)
excentricidades_2008 = calcular_excentricidade(adjacencia_2008)
diametro_2008 = calcular_diametro(adjacencia_2008)
centro_2008 = calcular_centro(adjacencia_2008)
vertices_2008 = sorted(adjacencia_2008)

metricas_2008 = pd.DataFrame({
    'Vertice': vertices_2008,
    'Ticker': [matriz_mantegna_formacao_2008.columns[v] for v in vertices_2008],
    'Grau': [graus_2008[v] for v in vertices_2008],
    'Excentricidade': [excentricidades_2008[v] for v in vertices_2008],
    'Folha': [graus_2008[v] == 1 for v in vertices_2008],
    'Centro': [v in centro_2008 for v in vertices_2008]
})

display(metricas_2008.sort_values(['Excentricidade', 'Ticker']))
print('Diametro de 2008:', diametro_2008)
print('Centro de 2008:', [matriz_mantegna_formacao_2008.columns[v] for v in centro_2008])


,Vertice,Ticker,Grau,Excentricidade,Folha,Centro
18,18,GGBR4.SA,4,4.523377,False,True
19,19,GOAU4.SA,1,5.039025,True,False
28,28,PETR4.SA,3,5.211030,False,False
17,17,GGBR3.SA,1,5.366267,True,False
34,34,USIM5.SA,11,5.391981,False,False
27,27,PETR3.SA,2,5.775584,False,False
12,12,CSNA3.SA,2,6.165553,False,False
33,33,USIM3.SA,1,6.327221,True,False
6,6,BRKM5.SA,4,6.347457,False,False
1,1,AXIA3.SA,3,6.351036,False,False


Diametro de 2008: 8.77770664822103
Centro de 2008: ['GGBR4.SA']


Selecionamos as oito menores excentricidades da MST de 2008 para a carteira central. O desempate continua sendo feito pelo ticker em ordem alfabética, sem consultar os retornos de 2007 a 2009.

In [53]:
# Mantem o mesmo criterio da carteira central da pandemia.
carteira_central_2008 = metricas_2008.sort_values(
    ['Excentricidade', 'Ticker'], ascending=[True, True]
).head(8).copy()

display(carteira_central_2008)


,Vertice,Ticker,Grau,Excentricidade,Folha,Centro
18,18,GGBR4.SA,4,4.523377,False,True
19,19,GOAU4.SA,1,5.039025,True,False
28,28,PETR4.SA,3,5.211030,False,False
17,17,GGBR3.SA,1,5.366267,True,False
34,34,USIM5.SA,11,5.391981,False,False
27,27,PETR3.SA,2,5.775584,False,False
12,12,CSNA3.SA,2,6.165553,False,False
33,33,USIM3.SA,1,6.327221,True,False


Selecionamos as oito maiores excentricidades da MST de 2008 para a carteira periférica. O desempate continua sendo feito pelo ticker em ordem alfabética e a seleção é independente da carteira central.

In [54]:
# Mantem o mesmo criterio da carteira periferica da pandemia.
carteira_periferica_2008 = metricas_2008.sort_values(
    ['Excentricidade', 'Ticker'], ascending=[False, True]
).head(8).copy()

display(carteira_periferica_2008)


,Vertice,Ticker,Grau,Excentricidade,Folha,Centro
4,4,BRAP3.SA,1,8.777707,True,False
21,21,ISAE4.SA,1,8.777707,True,False
9,9,CMIG4.SA,2,8.687644,False,False
13,13,EGIE3.SA,1,8.649321,True,False
20,20,INEP4.SA,1,8.548920,True,False
31,31,SBSP3.SA,1,8.391085,True,False
5,5,BRAP4.SA,2,7.615889,False,False
10,10,COCE5.SA,1,7.601276,True,False


O controle de 2008 adapta uma lista de pagadoras de dividendos publicada pela InfoMoney em 28/12/2006, antes da avaliação. Mantemos os tickers presentes na MST, preservamos a ordem da publicação e escolhemos até oito empresas, sem duas classes da mesma companhia e sem usar métricas do grafo; os segmentos são descrições de atividade. Essa lista não é uma carteira comprovadamente anticrise nem uma réplica da referência BTG de 2019. Fonte: https://www.infomoney.com.br/mercados/em-2006-quem-investiu-em-boas-pagadoras-de-proventos-nao-tem-do-que-reclamar/

In [55]:
# Referencia publicada em 28/12/2006, com informacoes ate 27/12/2006.
fonte_controle_2008 = 'https://www.infomoney.com.br/mercados/em-2006-quem-investiu-em-boas-pagadoras-de-proventos-nao-tem-do-que-reclamar/'
referencia_controle_2008 = pd.DataFrame([
    ('Telesp', 'TLPP4.SA', 'Telecom'),
    ('Cemig', 'CMIG3.SA', 'Energia eletrica'),
    ('CSN', 'CSNA3.SA', 'Siderurgia'),
    ('Cemig', 'CMIG4.SA', 'Energia eletrica'),
    ('Comgas', 'CGAS5.SA', 'Distribuicao de gas'),
    ('Banco do Brasil', 'BBAS3.SA', 'Bancos'),
    ('Celesc', 'CLSC6.SA', 'Energia eletrica'),
    ('Sabesp', 'SBSP3.SA', 'Saneamento'),
    ('BrT Participacoes', 'BRTP4.SA', 'Telecom'),
    ('Brasil Telecom', 'BRTO4.SA', 'Telecom'),
    ('Telemar Norte Leste', 'TMAR5.SA', 'Telecom'),
    ('Transmissao Paulista', 'TRPL4.SA', 'Energia eletrica'),
    ('Usiminas', 'USIM5.SA', 'Siderurgia'),
    ('Ipiranga Petroleo', 'PTIP4.SA', 'Petroleo e gas'),
    ('Acesita', 'ACES4.SA', 'Siderurgia'),
    ('Souza Cruz', 'CRUZ3.SA', 'Tabaco'),
    ('Unibanco', 'UBBR11.SA', 'Bancos'),
    ('Gerdau Metalurgica', 'GOAU4.SA', 'Siderurgia'),
    ('Telemig Participacoes', 'TMCP4.SA', 'Telecom'),
    ('Petrobras', 'PETR4.SA', 'Petroleo e gas')
], columns=['Empresa', 'Ticker', 'Segmento'])

# A MST e usada somente para verificar a presenca dos tickers.
tickers_mst_2008 = {
    matriz_mantegna_formacao_2008.columns[vertice]
    for _, u, v in mst_kruskal_2008
    for vertice in (u, v)
}
presentes_2008 = referencia_controle_2008['Ticker'].isin(tickers_mst_2008)
carteira_controle_2008 = referencia_controle_2008.loc[presentes_2008].copy()
# A ordem externa escolhe a classe quando ha duas acoes da mesma empresa.
carteira_controle_2008 = carteira_controle_2008.drop_duplicates('Empresa').head(8)
carteira_controle_2008 = carteira_controle_2008.reset_index(drop=True)

print('Referencia de 2008: lista de dividendos publicada em 28/12/2006')
print('Fonte:', fonte_controle_2008)
print('Papeis da referencia fora da MST:', referencia_controle_2008.loc[~presentes_2008, 'Ticker'].tolist())
display(carteira_controle_2008)


Referencia de 2008: lista de dividendos publicada em 28/12/2006
Fonte: https://www.infomoney.com.br/mercados/em-2006-quem-investiu-em-boas-pagadoras-de-proventos-nao-tem-do-que-reclamar/
Papeis da referencia fora da MST: ['TLPP4.SA', 'CLSC6.SA', 'BRTP4.SA', 'BRTO4.SA', 'TMAR5.SA', 'TRPL4.SA', 'PTIP4.SA', 'ACES4.SA', 'CRUZ3.SA', 'UBBR11.SA', 'TMCP4.SA']


,Empresa,Ticker,Segmento
0,Cemig,CMIG3.SA,Energia eletrica
1,CSN,CSNA3.SA,Siderurgia
2,Comgas,CGAS5.SA,Distribuicao de gas
3,Banco do Brasil,BBAS3.SA,Bancos
4,Sabesp,SBSP3.SA,Saneamento
5,Usiminas,USIM5.SA,Siderurgia
6,Gerdau Metalurgica,GOAU4.SA,Siderurgia
7,Petrobras,PETR4.SA,Petroleo e gas
